In [1]:
import json
import os, statistics, math

def extract_step_to_acc(path: str):
    """
    Read a raw metrics file and return {step: test_acc} from even-numbered lines only.
    - 1-based line numbering (keep only even lines)
    - JSON parse; keep only split == 'test'
    - Map: step -> acc (latest occurrence wins if duplicated)
    """
    result = {}
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):  # 1-based
            line = line.strip()
            if not line or (i % 2 != 0):  # skip odd lines
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get("split") != "test":
                continue
            step = obj.get("step")
            acc = obj.get("acc")
            if step is not None and acc is not None:
                result[step] = acc  # latest one wins
    return result

In [2]:
def summarize_by_step(root='.', start=1, end=53, prefix='client_', ext='.raw',
                      variance='population', step_keys=None, require_all=False):
    """
    Aggregate across clients PER STEP and return TWO dictionaries:
      - mean_by_step: {step: mean_test_acc_over_clients}
      - var_by_step:  {step: variance_test_acc_over_clients}

    Parameters:
      - variance: 'population' (statistics.pvariance) | 'sample' (statistics.variance)
      - step_keys: optional iterable of steps to enforce (e.g., [0,10,20,...,100]).
                   If None, uses the union of steps observed across clients.
      - require_all: if True, only include a step if ALL clients have a value for it.
                     If False, compute from available values.

    Notes:
      - Uses extract_step_to_acc(path) from earlier cell (even-line, split=='test').
      - Missing files or missing steps are ignored per 'require_all' policy.
    """
    # Collect per-step lists of accuracies across clients
    accs_by_step = {}
    client_count = 0
    for i in range(start, end + 1):
        path = os.path.join(root, f"{prefix}{i:03d}{ext}")
        try:
            d = extract_step_to_acc(path)
        except FileNotFoundError:
            # Missing client file; skip
            continue
        client_count += 1
        for step, acc in d.items():
            accs_by_step.setdefault(step, []).append(acc)

    # Decide which steps to include
    if step_keys is None:
        steps = sorted(accs_by_step.keys())
    else:
        steps = list(step_keys)

    mean_by_step = {}
    var_by_step  = {}
    for step in steps:
        vals = accs_by_step.get(step, [])
        if not vals:
            continue  # no data at this step
        if require_all and len(vals) != client_count:
            # Skip this step because not all clients provided it
            continue
        m = sum(vals) / len(vals)
        if variance == 'population':
            v = statistics.pvariance(vals)
        elif variance == 'sample':
            v = statistics.variance(vals) if len(vals) > 1 else float('nan')
        else:
            raise ValueError("variance must be 'population' or 'sample'")
        mean_by_step[step] = m
        var_by_step[step]  = v
    return mean_by_step, var_by_step

In [3]:
mean_dict, var_dict = summarize_by_step(root='.', start=1, end=53)
print(mean_dict)
print(var_dict)

{0: 0.8270000000000002, 10: 0.8095000000000001, 20: 0.8150000000000001, 30: 0.8205, 40: 0.8255000000000001, 50: 0.8255000000000001, 60: 0.8270000000000002, 70: 0.8265000000000002, 80: 0.8310000000000004, 90: 0.8365, 100: 0.8300000000000003, 110: 0.8419999999999999, 120: 0.8305, 130: 0.841, 140: 0.834, 150: 0.8415000000000002, 160: 0.8445, 170: 0.8465, 180: 0.8449999999999998, 190: 0.8414999999999999, 200: 0.8470000000000002, 210: 0.8485000000000001, 220: 0.842, 230: 0.8430000000000002, 240: 0.85, 250: 0.8435, 260: 0.8468421052631578, 270: 0.847027027027027, 280: 0.8513513513513514, 290: 0.8475675675675677, 300: 0.8438888888888889}
{0: 0.0042710000000000005, 10: 0.005379749999999998, 20: 0.003935, 30: 0.00352975, 40: 0.00415975, 50: 0.004119749999999999, 60: 0.004071, 70: 0.004327750000000001, 80: 0.004259, 90: 0.0033777499999999997, 100: 0.00312, 110: 0.0023960000000000006, 120: 0.0026997500000000008, 130: 0.002259, 140: 0.003404, 150: 0.00274775, 160: 0.00338975, 170: 0.00250775, 180:

In [4]:
mean_1_10, var_1_10 = summarize_by_step(root='.', start=1,  end=10)
mean_11_20, var_11_20 = summarize_by_step(root='.', start=11, end=20)
mean_21_30, var_21_30 = summarize_by_step(root='.', start=21, end=30)
mean_31_40, var_31_40 = summarize_by_step(root='.', start=31, end=40)

In [5]:
print(mean_1_10)
print(var_1_10)

{0: 0.842, 10: 0.8280000000000001, 20: 0.818, 30: 0.8200000000000001, 40: 0.8379999999999999, 50: 0.8260000000000002, 60: 0.8460000000000001, 70: 0.8300000000000001, 80: 0.8440000000000001, 90: 0.844, 100: 0.836, 110: 0.8500000000000002, 120: 0.8300000000000001, 130: 0.852, 140: 0.8380000000000001, 150: 0.85, 160: 0.858, 170: 0.8539999999999999, 180: 0.8560000000000001, 190: 0.842, 200: 0.8480000000000001, 210: 0.8540000000000001, 220: 0.85, 230: 0.86, 240: 0.858, 250: 0.8540000000000001, 260: 0.858, 270: 0.858, 280: 0.8560000000000001, 290: 0.8640000000000001, 300: 0.8488888888888889}
{0: 0.0012360000000000008, 10: 0.0025759999999999997, 20: 0.0025159999999999996, 30: 0.003120000000000001, 40: 0.0027560000000000006, 50: 0.002883999999999998, 60: 0.0012839999999999998, 70: 0.0009799999999999995, 80: 0.0038240000000000006, 90: 0.002224000000000001, 100: 0.000704, 110: 0.0010600000000000008, 120: 0.0021799999999999996, 130: 0.001856, 140: 0.0017159999999999996, 150: 0.0011400000000000004

In [6]:
print(mean_11_20)
print(var_11_20)

{0: 0.876, 10: 0.858, 20: 0.86, 30: 0.8620000000000001, 40: 0.8720000000000001, 50: 0.882, 60: 0.8699999999999999, 70: 0.8780000000000001, 80: 0.8780000000000001, 90: 0.8720000000000001, 100: 0.8719999999999999, 110: 0.8740000000000002, 120: 0.866, 130: 0.876, 140: 0.8800000000000001, 150: 0.874, 160: 0.8779999999999999, 170: 0.8800000000000001, 180: 0.8879999999999999, 190: 0.8859999999999999, 200: 0.8840000000000001, 210: 0.8760000000000001, 220: 0.8859999999999999, 230: 0.8699999999999999, 240: 0.8879999999999999, 250: 0.8879999999999999, 260: 0.8888888888888888, 270: 0.8844444444444446, 280: 0.8777777777777778, 290: 0.8733333333333335, 300: 0.8777777777777778}
{0: 0.001344000000000001, 10: 0.0007560000000000014, 20: 0.0010400000000000019, 30: 0.0013160000000000005, 40: 0.0012960000000000007, 50: 0.0008360000000000003, 60: 0.0004200000000000008, 70: 0.001156000000000002, 80: 0.0011560000000000008, 90: 0.0012960000000000022, 100: 0.0014560000000000011, 110: 0.0012040000000000022, 120

In [7]:
print(mean_21_30)
print(var_21_30)

{0: 0.8480000000000001, 10: 0.8320000000000001, 20: 0.834, 30: 0.8320000000000001, 40: 0.8320000000000001, 50: 0.8280000000000001, 60: 0.8379999999999999, 70: 0.844, 80: 0.842, 90: 0.8440000000000001, 100: 0.834, 110: 0.8460000000000001, 120: 0.8360000000000001, 130: 0.8340000000000002, 140: 0.8400000000000002, 150: 0.8460000000000001, 160: 0.8459999999999999, 170: 0.8400000000000001, 180: 0.8400000000000001, 190: 0.842, 200: 0.8340000000000002, 210: 0.8520000000000001, 220: 0.8400000000000002, 230: 0.8280000000000001, 240: 0.85, 250: 0.8320000000000001, 260: 0.8422222222222222, 270: 0.8250000000000001, 280: 0.8475, 290: 0.8525, 300: 0.8400000000000001}
{0: 0.0014560000000000005, 10: 0.0019359999999999976, 20: 0.003043999999999998, 30: 0.002576, 40: 0.002656, 50: 0.002655999999999998, 60: 0.0022760000000000002, 70: 0.0024640000000000005, 80: 0.002516, 90: 0.0031839999999999993, 100: 0.003363999999999998, 110: 0.0016840000000000002, 120: 0.0014240000000000001, 130: 0.0016039999999999993

In [8]:
print(mean_31_40)
print(var_31_40)

{0: 0.742, 10: 0.7200000000000001, 20: 0.748, 30: 0.768, 40: 0.76, 50: 0.766, 60: 0.754, 70: 0.7539999999999999, 80: 0.76, 90: 0.786, 100: 0.7779999999999999, 110: 0.7979999999999999, 120: 0.7899999999999999, 130: 0.8019999999999999, 140: 0.778, 150: 0.796, 160: 0.796, 170: 0.8119999999999999, 180: 0.796, 190: 0.796, 200: 0.8220000000000001, 210: 0.8119999999999999, 220: 0.792, 230: 0.8140000000000001, 240: 0.8039999999999999, 250: 0.7999999999999999, 260: 0.8019999999999999, 270: 0.8200000000000001, 280: 0.826, 290: 0.804, 300: 0.812}
{0: 0.0027560000000000024, 10: 0.005039999999999998, 20: 0.002256, 30: 0.002495999999999999, 40: 0.00328, 50: 0.0033639999999999994, 60: 0.004643999999999999, 70: 0.004483999999999998, 80: 0.001999999999999999, 90: 0.002883999999999999, 100: 0.002436, 110: 0.0025960000000000007, 120: 0.0030600000000000007, 130: 0.002035999999999999, 140: 0.003795999999999999, 150: 0.002304, 160: 0.003023999999999998, 170: 0.002096, 180: 0.0020640000000000003, 190: 0.0035